# Altas por día en el catálogo KEV

Cuaderno de lectura de la medición `kev/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Respalda los artículos [tres-dias-y-un-triaje](https://manpla.net/posts/tres-dias-y-un-triaje/) y [limpio-y-ya-estaba-dentro](https://manpla.net/posts/limpio-y-ya-estaba-dentro/). Carga el fichero de al lado —o lo descarga del repositorio si se ejecuta fuera de él—, muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no vuelve a tomar la instantánea: para eso está `generar.py`.

*Reading notebook for this measurement: loads the file next to it, prints the provenance record and draws one figure. Column names are in Spanish; `GLOSARIO.md` gives the English form.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/kev/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")


## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
altas = leer("altas.csv", parse_dates=["fecha"])
print(len(altas), "días con altas ·", altas.altas.sum(), "entradas")
altas.tail(15)

## Una figura

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4))
sem = altas.set_index("fecha").altas.resample("W").sum()
a.plot(sem.index, sem.values, lw=1); a.set_title("altas por semana"); a.set_ylabel("entradas")
orden = ["lunes","martes","miércoles","jueves","viernes","sábado","domingo"]
b.bar(orden, [altas[altas.dia_semana == d].altas.sum() for d in orden]); b.set_title("altas por día de la semana")
b.tick_params(axis="x", rotation=45); plt.tight_layout()

## La ventana, mes a mes

Cuántas altas tuvo cada mes y cuántas de ellas con ventana de 21, 14 o 3 días entre el alta y la fecha límite; y cuántas llevan la marca `forensicTriage`.

In [ ]:
meses = leer("ventanas-mensuales.csv")
meses.tail(12)

## Las entradas con marca de triaje forense

Una fila por entrada con `forensicTriage = Yes`: identificador, proveedor, producto, fechas, ventana, uso en ransomware, CWE y si el catálogo la anota como componente compartido.

In [ ]:
marca = leer("triaje-forense.csv", parse_dates=["fecha_alta", "vencimiento"])
print(len(marca), "entradas con marca ·", marca.ventana_dias.value_counts().to_dict(), "días de ventana")
print(marca.proveedor.value_counts().head(8))
marca.tail(10)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
sem = marca.set_index("fecha_alta").resample("W").size()
ax.bar(sem.index, sem.values, width=5, color="#e34948")
ax.set_title("altas con marca de triaje forense, por semana"); ax.set_ylabel("entradas")
plt.show()